# Augmented RAG with CauseNet + PubMed

This notebook demonstrates how to use the enhanced AugmentedModelSuggester that combines:
- **CauseNet**: Structured causal knowledge base
- **PubMed**: Scientific literature abstracts

The system automatically:
1. Searches CauseNet for matching causal pairs
2. Queries PubMed with query rewriting
3. Combines both sources into unified retriever
4. Generates LLM response with augmented context

## Setup

In [ ]:
import sys
import os
import time
import pandas as pd


# Add the notebook setup path to sys.path
notebook_setup_path = os.path.abspath("../")
sys.path.insert(0, notebook_setup_path)

from notebook_setup import setup_local_pywhyllm
project_root = setup_local_pywhyllm()

### Tuebingen dataset   


In [2]:
df = pd.read_csv('/home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/tuebingen_pairs.csv')


In [7]:
from dotenv import load_dotenv
import guidance
from openai import OpenAI
from portkey_ai import createHeaders
from langchain_openai import ChatOpenAI

load_dotenv()

#azure_model="gpt-4.1-2025-04-14"
azure_model= "o4-mini-2025-04-16"
us_base_url = "https://us.aigw.galileo.roche.com/v1"

portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])

# For guidance (used by SimpleModelSuggester methods)
model = guidance.models.OpenAI(
    azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers
)


langchain_llm = ChatOpenAI(
    model=azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers,
    #temperature=0
)

In [8]:
from pywhyllm.suggesters.augmented_model_suggester_alba import AugmentedModelSuggester

causenet_path ="/home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/data/causenet-precision.jsonl.bz2"
# Initialize with both Guidance and LangChain LLMs
suggester = AugmentedModelSuggester(
    llm=model,  # Guidance model for simple suggester methods
    langchain_llm=langchain_llm,  # LangChain model for RAG queries
    pubmed_email="albamaria.molero.perez@example.com",  
    file_path=causenet_path  # Use existing local file
)


✓ CauseNet found locally at /home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/data/causenet-precision.jsonl.bz2
Loading CauseNet using json
Done loading CauseNet using json
Creating dictionary from CauseNet json data
Done creating dictionary from CauseNet json data


In [9]:
from typing import Dict, List, Tuple
llm_output : Dict[str, dict] = {}

# Define parameters for the experiment
temperature = 0
num_runs = 1 # Reduced for testing

# Create saved_pairs_info to store ground truth and variable info
saved_pairs_info = {}

# Process only the first 3 rows for testing
#test_df = df.head(1) #or just df all dataset
test_df = df #or just df all dataset
#test_df = df.iloc[50:]  


# Iterate through each pair and run multiple times
start = time.time()

for pair_number, values in test_df.iterrows():
    pair_id = f"pair{pair_number:04d}"  # Create pair ID like "pair0001", "pair0002", etc.

    var1_value = values['var1'].strip() if isinstance(values['var1'], str) else values['var1']
    var2_value = values['var2'].strip() if isinstance(values['var2'], str) else values['var2']
    ground_truth_value = values['ground_truth']
    if isinstance(ground_truth_value, str):
        ground_truth_value = ground_truth_value.strip().upper()

    saved_pairs_info[pair_id] = {
        "var1": var1_value,
        "var2": var2_value,
        "ground_truth": ground_truth_value,  # columna en tu dataframe
        "truth_ab": int(values['truth_ab']),
        "truth_ba": int(values['truth_ba'])
    }
    
    for n in range(1, num_runs + 1):  # Run 1 to 5
        temp_dict = {}
        
        print(f"Processing {pair_id}, run {n}/{num_runs}")

        temp_dict['llm_ab'] = suggester.suggest_pairwise_relationship(
        variable1=var1_value, 
        variable2=var2_value, 
        use_pubmed=True,
        max_pubmed_papers=5,
        return_prompt=True, 
        confidence_level=False
)
                # )
        temp_dict['llm_ba'] = suggester.suggest_pairwise_relationship(
        variable1=var2_value, 
        variable2=var1_value, 
        use_pubmed=True,
        max_pubmed_papers=5,
        return_prompt=True, 
        confidence_level=False
)
        # Store results with key: (pair_id, temperature, run_number)
        llm_output[(pair_id, temperature, n )] = temp_dict
        
        print(f"  A->B: {temp_dict['llm_ab']}, B->A: {temp_dict['llm_ba']}")

# Calculate latencies after all processing is complete
total_time = time.time() - start
print(f"\nCompleted processing {len(test_df)} pairs with {num_runs} runs each")
print(f"Total execution time: {total_time:.2f}s")

# Calculate latencies
avg_latency_per_pair = total_time / (len(test_df) * num_runs)  # Time per pair (both A->B and B->A) per run
avg_latency_per_run = total_time / (len(test_df) * num_runs * 2)  # Time per individual query (A->B or B->A)

print(f"Average time per pair (A->B + B->A): {avg_latency_per_pair:.2f}s")
print(f"Average time per individual query: {avg_latency_per_run:.2f}s")

Processing pair0000, run 1/1

🔬 Analyzing: Altitude ↔ Temperature

📖 Step 1: Searching CauseNet...


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

✗ No CauseNet match above threshold 0.35 (Best: 0.1573 for 'autoimmune_gfap_astrocytopathy-fever')
   ✗ No CauseNet match found

📚 Step 2: Searching PubMed...
🔍 Searching PubMed for: temperature ↔ time for 1/6 rotation
   Trying query: temperature AND time for 1/6 rotation AND (causal OR causation OR cause)


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


   ✓ Found 5 papers
📚 Total papers retrieved: 5
   ✓ Retrieved PubMed literature (text length: 7921 chars)

🤖 Step 3: Querying LLM...
📊 Creating combined retriever (text length: 7960 chars)


INFO:backoff:Backing off send_request(...) for 0.5s (requests.exceptions.SSLError: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)'))))
INFO:backoff:Backing off send_request(...) for 0.1s (requests.exceptions.SSLError: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)'))))
INFO:backoff:Backing off send_request(...) for 3.3s (requests.exceptions.SSLError: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed

   LLM response generated with RAG augmentation

 Step 4: Parsing result...
   → Causal direction: temperature → time for 1/6 rotation

  A->B: {'result': ['temperature', 'time for 1/6 rotation', 'My reasoning:\n\n1. A variable “time for 1/6 rotation” describes a mechanical period or speed.  \n2. Ambient temperature can affect mechanical properties (e.g. lubricants’ viscosity, friction in bearings), thereby altering the rotation time.  \n3. Conversely, the time it takes to rotate 1/6 of a turn would not meaningfully alter ambient temperature.  \n4. Therefore, temperature is the cause and rotation‐time is the effect.\n\n<answer>B</answer>'], 'system_prompt': "You are a helpful assistant for causal reasoning.\n\n    Context: Eight winter air pollution-cold wave sequential events were identified over 4 years, with higher daily IS incidence during event periods (67 new cases/day), lag periods (68 new cases/day) than non-event periods (60 new cases/day). Subgroup analysis showed that among 

In [10]:
len(llm_output)

108

In [11]:
azure_model

'o4-mini-2025-04-16'

In [ ]:
# import json
# from pathlib import Path
# from datetime import datetime

# # Create a timestamp for unique filenames
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# json_filename = f"llm_output_rag_method_last50pairs{timestamp}.json"

# # Convert tuple keys to strings for JSON compatibility
# llm_output_serializable = {
#     f"{pair_id}_{temp}_{run}": value 
#     for (pair_id, temp, run), value in llm_output.items()
# }

# # Save to JSON
# with open(json_filename, 'w', encoding='utf-8') as f:
#     json.dump(llm_output_serializable, f, indent=2, ensure_ascii=False)

# Function to load the data back in the original format
def load_llm_output(filename):
    """
    Load llm_output from saved JSON file.
    """
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Convert string keys back to tuples
    result = {}
    for key_str, value in data.items():
        parts = key_str.rsplit('_', 2)
        pair_id = parts[0]
        temp = float(parts[1])
        run = int(parts[2])
        result[(pair_id, temp, run)] = value
    
    return result


In [12]:
loaded_data = load_llm_output('/home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/llm_output_rag_method_20251103_215347_first50.json')

In [ ]:
llm_output= loaded_data


Export results from Tuebingen causality pairs dataset to CSV for processing.

In [12]:
results : Dict = {}

# Helper utilities to compare LLM output against dataset direction
def _normalize_term(value):
    if isinstance(value, str):
        return value.strip().lower()
    return value

def _infer_direction(result_list, var1, var2):
    if not result_list:
        return "UNKNOWN"
    cause, effect = result_list[0], result_list[1]
    if cause is None or effect is None:
        return "NONE"
    cause_norm = _normalize_term(cause)
    effect_norm = _normalize_term(effect)
    v1_norm = _normalize_term(var1)
    v2_norm = _normalize_term(var2)
    if cause_norm == v1_norm and effect_norm == v2_norm:
        return "R"
    if cause_norm == v2_norm and effect_norm == v1_norm:
        return "L"
    return "UNKNOWN"

for pair_id, info in saved_pairs_info.items():
    av_correct_ab = 0
    av_correct_ba = 0
    
    # Lists to collect metrics across runs
    confidence_scores_ab = []
    confidence_scores_ba = []
    strength_scores_ab = []
    strength_scores_ba = []
    logprobs_ab = []
    logprobs_ba = []
    answer_choice_logprobs_ab = []
    answer_choice_logprobs_ba = []
    direction_runs_ab = []
    direction_runs_ba = []
    
    gt = info['ground_truth'] if isinstance(info['ground_truth'], str) else str(info['ground_truth'])
    gt = gt.strip().upper()

    for i in range(num_runs):
        key = (pair_id, temperature, i+1)
        ab_result = llm_output[key]['llm_ab']
        ba_result = llm_output[key]['llm_ba']
        
        # Get the result list from the dict
        ab_list = ab_result['result']  # [cause, effect, response]
        ba_list = ba_result['result']  # [cause, effect, response]
        
        # Extract confidence and strength scores
        ab_confidence = ab_result.get('confidence_score')
        ba_confidence = ba_result.get('confidence_score')
        ab_strength = ab_result.get('strength_score')
        ba_strength = ba_result.get('strength_score')
        
        # Extract logprobs data (keep individual values, don't average)
        ab_token_logprob = ab_result.get('answer_token_logprob')
        ba_token_logprob = ba_result.get('answer_token_logprob')
        ab_choice_logprobs = ab_result.get('answer_choice_logprobs', {})
        ba_choice_logprobs = ba_result.get('answer_choice_logprobs', {})
        
        # Store metrics (only if not None)
        if ab_confidence is not None:
            confidence_scores_ab.append(ab_confidence)
        if ba_confidence is not None:
            confidence_scores_ba.append(ba_confidence)
        if ab_strength is not None:
            strength_scores_ab.append(ab_strength)
        if ba_strength is not None:
            strength_scores_ba.append(ba_strength)
        if ab_token_logprob is not None:
            logprobs_ab.append(ab_token_logprob)
        if ba_token_logprob is not None:
            logprobs_ba.append(ba_token_logprob)
        if ab_choice_logprobs:
            answer_choice_logprobs_ab.append(ab_choice_logprobs)
        if ba_choice_logprobs:
            answer_choice_logprobs_ba.append(ba_choice_logprobs)
        
        # Determine predicted directions
        pred_ab_direction = _infer_direction(ab_list, info['var1'], info['var2'])
        pred_ba_direction = _infer_direction(ba_list, info['var1'], info['var2'])  
        direction_runs_ab.append(pred_ab_direction)
        direction_runs_ba.append(pred_ba_direction)
        
        # Evaluate correctness for "Does A cause B?"
        if pred_ab_direction == "R" and gt == "R":
            av_correct_ab += 1
        elif pred_ab_direction == "L" and gt == "L":
            av_correct_ab += 1
        
        # Evaluate correctness for "Does B cause A?"
        if pred_ba_direction == "L" and gt == "L":
            av_correct_ba += 1
        elif pred_ba_direction == "R" and gt == "R":
            av_correct_ba += 1

    # Calculate averages
    av_correct_ab /= num_runs
    av_correct_ba /= num_runs
    
    avg_confidence_ab = sum(confidence_scores_ab) / len(confidence_scores_ab) if confidence_scores_ab else None
    avg_confidence_ba = sum(confidence_scores_ba) / len(confidence_scores_ba) if confidence_scores_ba else None
    avg_strength_ab = sum(strength_scores_ab) / len(strength_scores_ab) if strength_scores_ab else None
    avg_strength_ba = sum(strength_scores_ba) / len(strength_scores_ba) if strength_scores_ba else None

    temp : Dict = {}
    temp['PairID'] = pair_id
    temp['CorrectACauseB'] = av_correct_ab
    temp['CorrectBCauseA'] = av_correct_ba
    temp['VarA'] = info['var1']
    temp['VarB'] = info['var2']
    temp['GroundTruth'] = gt
    temp['ConfidenceAB'] = avg_confidence_ab
    temp['ConfidenceBA'] = avg_confidence_ba
    temp['StrengthAB'] = avg_strength_ab
    temp['StrengthBA'] = avg_strength_ba
    temp['LogprobsAB'] = logprobs_ab  # Store list of logprobs (not averaged)
    temp['LogprobsBA'] = logprobs_ba  # Store list of logprobs (not averaged)
    temp['ChoiceLogprobsAB'] = answer_choice_logprobs_ab
    temp['ChoiceLogprobsBA'] = answer_choice_logprobs_ba
    temp['PredictedDirectionAB'] = direction_runs_ab
    temp['PredictedDirectionBA'] = direction_runs_ba

    results[pair_id] = temp
    print(results[pair_id])


{'PairID': 'pair0000', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': 'Altitude', 'VarB': 'Temperature', 'GroundTruth': 'R', 'ConfidenceAB': None, 'ConfidenceBA': None, 'StrengthAB': None, 'StrengthBA': None, 'LogprobsAB': [], 'LogprobsBA': [], 'ChoiceLogprobsAB': [], 'ChoiceLogprobsBA': [], 'PredictedDirectionAB': ['R'], 'PredictedDirectionBA': ['R']}
{'PairID': 'pair0001', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': 'Altitude', 'VarB': 'Precipitation', 'GroundTruth': 'R', 'ConfidenceAB': None, 'ConfidenceBA': None, 'StrengthAB': None, 'StrengthBA': None, 'LogprobsAB': [], 'LogprobsBA': [], 'ChoiceLogprobsAB': [], 'ChoiceLogprobsBA': [], 'PredictedDirectionAB': ['R'], 'PredictedDirectionBA': ['R']}
{'PairID': 'pair0002', 'CorrectACauseB': 0.0, 'CorrectBCauseA': 0.0, 'VarA': 'Longitude', 'VarB': 'Temperature', 'GroundTruth': 'R', 'ConfidenceAB': None, 'ConfidenceBA': None, 'StrengthAB': None, 'StrengthBA': None, 'LogprobsAB': [], 'LogprobsBA': [], 'ChoiceLogprobsAB': [

In [13]:
# Calculate accuracy metrics (excluding mean logprobs)
accuracy_results = {}

total_pairs = len(results)
sum_ab_accuracy = 0
sum_ba_accuracy = 0
sum_joint_accuracy = 0
sum_ab_confidence = 0
sum_ba_confidence = 0
sum_ab_strength = 0
sum_ba_strength = 0
count_ab_confidence = 0
count_ba_confidence = 0
count_ab_strength = 0
count_ba_strength = 0

for pair_id, result in results.items():
    correct_ab = result['CorrectACauseB']
    correct_ba = result['CorrectBCauseA']
    
    # Get all metrics (excluding logprobs from averaging)
    confidence_ab = result.get('ConfidenceAB')
    confidence_ba = result.get('ConfidenceBA')
    strength_ab = result.get('StrengthAB')
    strength_ba = result.get('StrengthBA')
    logprobs_ab = result.get('LogprobsAB', [])  # Keep as list
    logprobs_ba = result.get('LogprobsBA', [])  # Keep as list
    
    joint_accuracy = (correct_ab + correct_ba) / 2.0
    
    sum_ab_accuracy += correct_ab
    sum_ba_accuracy += correct_ba
    sum_joint_accuracy += joint_accuracy
    
    # Sum confidence scores
    if confidence_ab is not None:
        sum_ab_confidence += confidence_ab
        count_ab_confidence += 1
    if confidence_ba is not None:
        sum_ba_confidence += confidence_ba
        count_ba_confidence += 1
    
    # Sum strength scores
    if strength_ab is not None:
        sum_ab_strength += strength_ab
        count_ab_strength += 1
    if strength_ba is not None:
        sum_ba_strength += strength_ba
        count_ba_strength += 1
    
    # Store individual pair results
    accuracy_results[pair_id] = {
        'PairID': pair_id,
        'VarA': result['VarA'],
        'VarB': result['VarB'],
        'GroundTruth': result['GroundTruth'],
        'AccuracyAB': correct_ab,
        'AccuracyBA': correct_ba,
        'JointAccuracy': joint_accuracy,
        'ConfidenceAB': confidence_ab,
        'ConfidenceBA': confidence_ba,
        'StrengthAB': strength_ab,
        'StrengthBA': strength_ba,
        'LogprobsAB': logprobs_ab,  # Store as list (not averaged)
        'LogprobsBA': logprobs_ba   # Store as list (not averaged)
    }
    
    pred_dirs_ab = result.get('PredictedDirectionAB', [])
    pred_dirs_ba = result.get('PredictedDirectionBA', [])
    
    # Display individual logprobs instead of mean
    logprob_ab_str = f"{logprobs_ab[0]:.4f}" if logprobs_ab else 'N/A'
    logprob_ba_str = f"{logprobs_ba[0]:.4f}" if logprobs_ba else 'N/A'
    
    print(f"Pair {pair_id}: {result['VarA']} -> {result['VarB']}")
    print(f"  Ground Truth: {result['GroundTruth']}")
    print(f"  A→B Accuracy: {correct_ab:.3f}, Confidence: {f'{confidence_ab:.3f}' if confidence_ab else 'N/A'}, Strength: {f'{strength_ab:.3f}' if strength_ab else 'N/A'}, Logprob: {logprob_ab_str}")
    print(f"  B→A Accuracy: {correct_ba:.3f}, Confidence: {f'{confidence_ba:.3f}' if confidence_ba else 'N/A'}, Strength: {f'{strength_ba:.3f}' if strength_ba else 'N/A'}, Logprob: {logprob_ba_str}")
    print(f"  Predicted directions AB runs: {pred_dirs_ab}")
    print(f"  Predicted directions BA runs: {pred_dirs_ba}")
    print(f"  Joint Accuracy (avg): {joint_accuracy:.3f}")
    print()

# Overall statistics (without mean logprobs)
overall_ab_accuracy = sum_ab_accuracy / total_pairs
overall_ba_accuracy = sum_ba_accuracy / total_pairs
overall_joint_accuracy = sum_joint_accuracy / total_pairs
overall_ab_confidence = sum_ab_confidence / count_ab_confidence if count_ab_confidence > 0 else None
overall_ba_confidence = sum_ba_confidence / count_ba_confidence if count_ba_confidence > 0 else None
overall_ab_strength = sum_ab_strength / count_ab_strength if count_ab_strength > 0 else None
overall_ba_strength = sum_ba_strength / count_ba_strength if count_ba_strength > 0 else None

print("=== OVERALL ACCURACY STATISTICS ===")
print(f"Total pairs processed: {total_pairs}")
print(f"\nMEAN ACCURACIES (across all pairs):")
print(f"A→B Mean Accuracy: {overall_ab_accuracy:.3f}")
print(f"B→A Mean Accuracy: {overall_ba_accuracy:.3f}")
print(f"Joint Mean Accuracy: {overall_joint_accuracy:.3f}")
print(f"\nMEAN CONFIDENCE SCORES:")
print(f"A→B Mean Confidence: {f'{overall_ab_confidence:.3f}' if overall_ab_confidence else 'N/A'}")
print(f"B→A Mean Confidence: {f'{overall_ba_confidence:.3f}' if overall_ba_confidence else 'N/A'}")
print(f"\nMEAN STRENGTH SCORES:")
print(f"A→B Mean Strength: {f'{overall_ab_strength:.3f}' if overall_ab_strength else 'N/A'}")
print(f"B→A Mean Strength: {f'{overall_ba_strength:.3f}' if overall_ba_strength else 'N/A'}")
print(f"\nLATENCY METRICS:")
print(f"Average time per pair (both directions): {avg_latency_per_pair:.2f}s")
print(f"Average time per query: {avg_latency_per_run:.2f}s")

# Summary statistics (without mean logprobs)
accuracy_summary = {
    'total_pairs': total_pairs,
    'ab_mean_accuracy': overall_ab_accuracy,
    'ba_mean_accuracy': overall_ba_accuracy,
    'joint_mean_accuracy': overall_joint_accuracy,
    'ab_mean_confidence': overall_ab_confidence,
    'ba_mean_confidence': overall_ba_confidence,
    'ab_mean_strength': overall_ab_strength,
    'ba_mean_strength': overall_ba_strength,
    'avg_latency_per_pair_seconds': round(avg_latency_per_pair, 2),
    'avg_latency_per_query_seconds': round(avg_latency_per_run, 2),
    'total_execution_time_seconds': round(total_time, 2)
}


Pair pair0000: Altitude -> Temperature
  Ground Truth: R
  A→B Accuracy: 1.000, Confidence: N/A, Strength: N/A, Logprob: N/A
  B→A Accuracy: 1.000, Confidence: N/A, Strength: N/A, Logprob: N/A
  Predicted directions AB runs: ['R']
  Predicted directions BA runs: ['R']
  Joint Accuracy (avg): 1.000

Pair pair0001: Altitude -> Precipitation
  Ground Truth: R
  A→B Accuracy: 1.000, Confidence: N/A, Strength: N/A, Logprob: N/A
  B→A Accuracy: 1.000, Confidence: N/A, Strength: N/A, Logprob: N/A
  Predicted directions AB runs: ['R']
  Predicted directions BA runs: ['R']
  Joint Accuracy (avg): 1.000

Pair pair0002: Longitude -> Temperature
  Ground Truth: R
  A→B Accuracy: 0.000, Confidence: N/A, Strength: N/A, Logprob: N/A
  B→A Accuracy: 0.000, Confidence: N/A, Strength: N/A, Logprob: N/A
  Predicted directions AB runs: ['NONE']
  Predicted directions BA runs: ['NONE']
  Joint Accuracy (avg): 0.000

Pair pair0003: Altitude -> Sunshine hours
  Ground Truth: R
  A→B Accuracy: 1.000, Confiden

In [14]:
import csv
import json

# CSV file for detailed accuracy results
accuracy_csv_file = "alba_accuracy_results_m2_o4mini.csv"

# Updated headers (logprobs stored as JSON strings in CSV)
accuracy_header = [
    "PairID", "VarA", "VarB", "GroundTruth", 
    "AccuracyAB", "AccuracyBA", "JointAccuracy",
    "ConfidenceAB", "ConfidenceBA",
    "StrengthAB", "StrengthBA",
    "LogprobsAB", "LogprobsBA"  
]

# Write detailed accuracy results (convert logprobs lists to JSON strings)
with open(accuracy_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=accuracy_header)
    writer.writeheader()
    for pair_id, values in accuracy_results.items():
        # Convert logprobs lists to JSON strings for CSV storage
        row = values.copy()
        row['LogprobsAB'] = json.dumps(values.get('LogprobsAB', []))
        row['LogprobsBA'] = json.dumps(values.get('LogprobsBA', []))
        writer.writerow(row)

print(f"Detailed accuracy CSV file '{accuracy_csv_file}' has been created.")

# CSV file for summary statistics
summary_csv_file = "alba_accuracy_summary_method2_o4mini.csv"

# Write summary statistics
with open(summary_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(accuracy_summary.keys()))
    writer.writeheader()
    writer.writerow(accuracy_summary)

print(f"Summary accuracy CSV file '{summary_csv_file}' has been created.")

# Display final summary table (without mean logprobs)
print("\n=== FINAL SUMMARY TABLE ===")
print(f"{'Metric':<25} {'Mean Accuracy':<15} {'Mean Confidence':<18} {'Mean Strength':<15}")
print("-" * 80)

ab_conf_str = f"{accuracy_summary['ab_mean_confidence']:.3f}" if accuracy_summary['ab_mean_confidence'] else 'N/A'
ba_conf_str = f"{accuracy_summary['ba_mean_confidence']:.3f}" if accuracy_summary['ba_mean_confidence'] else 'N/A'
ab_str_str = f"{accuracy_summary['ab_mean_strength']:.3f}" if accuracy_summary['ab_mean_strength'] else 'N/A'
ba_str_str = f"{accuracy_summary['ba_mean_strength']:.3f}" if accuracy_summary['ba_mean_strength'] else 'N/A'

print(f"{'A→B':<25} {accuracy_summary['ab_mean_accuracy']:<15.3f} {ab_conf_str:<18} {ab_str_str:<15}")
print(f"{'B→A':<25} {accuracy_summary['ba_mean_accuracy']:<15.3f} {ba_conf_str:<18} {ba_str_str:<15}")
print(f"{'Joint Accuracy':<25} {accuracy_summary['joint_mean_accuracy']:<15.3f} {'N/A':<18} {'N/A':<15}")
print(f"{'Total Pairs':<25} {accuracy_summary['total_pairs']:<15} {'N/A':<18} {'N/A':<15}")
print("\nNote: Individual logprobs per pair are stored in the detailed CSV file.")


Detailed accuracy CSV file 'alba_accuracy_results_m2_o4mini.csv' has been created.
Summary accuracy CSV file 'alba_accuracy_summary_method2_o4mini.csv' has been created.

=== FINAL SUMMARY TABLE ===
Metric                    Mean Accuracy   Mean Confidence    Mean Strength  
--------------------------------------------------------------------------------
A→B                       0.880           N/A                N/A            
B→A                       0.824           N/A                N/A            
Joint Accuracy            0.852           N/A                N/A            
Total Pairs               108             N/A                N/A            

Note: Individual logprobs per pair are stored in the detailed CSV file.
